# Weather-or-Not: Flight Delay Classifier
### Naive Bayes vs. Feedforward Neural Network

**Target:** Predict whether a flight will arrive delayed (≥15 min) based on weather at origin & destination airports.

**Pipeline:**
1. Load flight + weather data
2. Merge weather onto flights (origin & destination)
3. Feature engineering
4. Train / test split
5. Model with FFN


## 0. Imports & Config

In [36]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import holidays
from datetime import timedelta

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 50)

with open('params_file.json') as f:
    params = json.load(f)

FLIGHT_CSV      = params['Output_Directory'] + 'flights_dataset.csv'
IEM_WEATHER_DIR = params['Dataset_Directory'] + '2024_iem_weather'
DELAY_THRESHOLD = 15        # FAA standard: 15+ min = delayed
RANDOM_STATE    = 1354
TEST_SIZE       = 0.2

## 1. Load Data

In [3]:
%%time
#Same loading as with NB
CACHE_PATH = 'flights_with_weather.parquet'

if os.path.exists(CACHE_PATH):
    print('Loading cached merged dataset...')
    merged = pd.read_parquet(CACHE_PATH)
else:
    print('Joining weather onto flights (this may take a few minutes)...')
    weather_records = []
    for _, row in flights.iterrows():
        orig_wx = get_interpolated_weather(row.get('ORIGIN',''), row['DEP_TS'], iem_weather, 'ORIG')
        dest_wx = get_interpolated_weather(row.get('DEST',''),   row['ARR_TS'], iem_weather, 'DEST')
        weather_records.append({**orig_wx, **dest_wx})
    wx_df  = pd.DataFrame(weather_records, index=flights.index)
    merged = pd.concat([flights, wx_df], axis=1)
    merged.to_parquet(CACHE_PATH, index=False)
    print(f'Cached to {CACHE_PATH}')

print(f'Merged shape: {merged.shape}')

Loading cached merged dataset...
Merged shape: (6510337, 58)
CPU times: total: 6.91 s
Wall time: 2.26 s


## 4. Feature Engineering

In [24]:
# Temporal features
WEATHER_COLS = ['tmpf', 'dwpf', 'relh', 'drct', 'sknt', 'vsby', 'mslp', 'gust']

merged['DEP_HOUR']   = merged['DEP_TS'].dt.hour
merged['DEP_DOW']    = merged['DEP_TS'].dt.dayofweek
merged['DEP_MONTH']  = merged['DEP_TS'].dt.month
merged['IS_WEEKEND'] = (merged['DEP_DOW'] >= 5).astype(int)
merged['RUSH_HOUR']  = merged['DEP_HOUR'].apply(
    lambda h: 1 if h in range(7, 10) or h in range(16, 20) else 0
)

#Grab next holdays distance
def get_days_to_next_holiday(df, timestamp_col='DEP_TS'):
    print(df[timestamp_col][0])
    print(pd.to_datetime(df[timestamp_col][0]))
    df[timestamp_col] = pd.to_datetime(df[timestamp_col])
    #only 2024 for our dataset but good ot be sure
    years = df[timestamp_col].dt.year.unique()
    #only US holidays as this is a US databse
    us_holidays = holidays.US(years=list(years) + [max(years) + 1])
    holiday_dates = sorted(us_holidays.keys())
    holiday_series = pd.to_datetime(holiday_dates)

    def find_next(ts):
        future_holidays = holiday_series[holiday_series >= ts.normalize()]
        if not future_holidays.empty:
            #grab closest holiday in the future
            next_h = future_holidays[0]
            return (next_h - ts).days
        return np.nan

    return merged[timestamp_col].apply(find_next)

#Flight Denisty Funciton (will be slightly difficult to get from flight aware API)
def add_flight_density(df):
    #Get the orign/
    df = df.sort_values(['ORIGIN', 'DEP_TS'])
    
    #Grab flights within a 2 hour window at that airport
    density = (
        df.set_index('DEP_TS')
        .groupby('ORIGIN')['FLIGHTS'] # 'FLIGHTS' column is usually just 1s
        .rolling('2h', center=True)
        .count()
        .reset_index(drop=True)
    )
    
    df['FLIGHT_DENSITY'] = density.values
    return df

# Feature lists
CONTINUOUS_FEATURES = [
    'ORIG_tmpf','ORIG_dwpf','ORIG_relh','ORIG_sknt','ORIG_vsby','ORIG_mslp',
    'DEST_tmpf','DEST_dwpf','DEST_relh','DEST_sknt','DEST_vsby','DEST_mslp',
    'DELTA_tmpf','DELTA_sknt','DELTA_vsby',
    'DEP_HOUR', 'DAYS_TO_HOLIDAY', 'FLIGHT_DENSITY'
]

#Theese will need embeddings
CATEGORICAL_FEATURES = [
    'DEP_DOW','DEP_MONTH','IS_WEEKEND','RUSH_HOUR',
    'ORIG_LOW_VIS','ORIG_HIGH_WIND','ORIG_GUSTING',
    'DEST_LOW_VIS','DEST_HIGH_WIND','DEST_GUSTING',
]
ALL_FEATURES = CONTINUOUS_FEATURES + CATEGORICAL_FEATURES
TARGET = 'DELAYED'

CACHE_PATH = 'flights_with_weather_features.parquet'

if os.path.exists(CACHE_PATH):
    print('Loading cached merged dataset...')
    merged = pd.read_parquet(CACHE_PATH)
else:
    # Apply to dataframe
    print("Days To Holiday Feature:")
    %time
    merged['DAYS_TO_HOLIDAY'] = get_days_to_next_holiday(merged, 'DEP_TS')
    print("Flight Density Feature:")
    %time
    merged = add_flight_density(merged)

    print("Weather Delta Features")
    # Weather delta (destination - origin)
    for col in WEATHER_COLS:
        o, d = f'ORIG_{col}', f'DEST_{col}'
        if o in merged.columns and d in merged.columns:
            merged[f'DELTA_{col}'] = merged[d] - merged[o]

    print("Weather Flag Features")
    # Derived flags
    for pfx in ['ORIG', 'DEST']:
        merged[f'{pfx}_LOW_VIS']   = (merged[f'{pfx}_vsby'] < 3).astype(float)
        merged[f'{pfx}_HIGH_WIND'] = (merged[f'{pfx}_sknt'] > 20).astype(float)
        merged[f'{pfx}_GUSTING']   = merged[f'{pfx}_gust'].notna().astype(float)
    #Null Handling
    for feature in CATEGORICAL_FEATURES:
        merged[feature].fillna(-1, inplace = True)
    for feature in CONTINUOUS_FEATURES:
        #Mean imputation for now
        merged[feature].fillna(merged[feature].mean(), inplace = True)

merged.to_parquet(CACHE_PATH, index=False)
print(f'Cached to {CACHE_PATH}')




Days To Holiday Feature:
CPU times: total: 0 ns
Wall time: 362 μs
2024-01-01 05:22:00
2024-01-01 05:22:00
Flight Density Feature:
CPU times: total: 0 ns
Wall time: 4.29 μs
Weather Delta Features
Weather Flag Features
Cached to flights_with_weather_features.parquet


## 5. Output Labels

In [25]:
merged['WEATHER_DELAY'].unique()
merged['ARR_DELAY_GROUP'].clip(lower=0, upper=5, inplace=True)

merged['ARR_DELAY_GROUP'].fillna(0, inplace = True)

print(merged['ARR_DELAY_GROUP'].unique())

TARGET = 'ARR_DELAY_GROUP'

[0. 1. 5. 2. 3. 4.]


In [26]:
model_df = merged.copy()
print(f'Modelling dataset: {len(model_df):,} rows | Delay rate: {model_df[TARGET].mean():.1%}')

if 'CANCELLED' in model_df.columns:
    model_df = model_df[model_df['CANCELLED'] == 0]
if 'DIVERTED' in model_df.columns:
    model_df = model_df[model_df['DIVERTED'] == 0]
model_df = model_df[ALL_FEATURES + [TARGET]].dropna(subset=[TARGET])

Modelling dataset: 6,510,337 rows | Delay rate: 56.7%


## 6. Train / Test Split

In [ ]:

#Scale Continuous columns and use one-hot for catagorical
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), CONTINUOUS_FEATURES), 
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES)
    ])

#FFN Def
ffn = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu', 
    solver='adam',
    max_iter=500,
    scoring=f1_macro_scorer,
    early_stopping=True,
    random_state=5
)

#pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', ffn)
])

print("Pipeline constructed.")

Pipeline constructed.


In [38]:
#X
X = model_df[CONTINUOUS_FEATURES + CATEGORICAL_FEATURES]
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
#Label Encoder, not that imporntant as delay groups are already integers
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(model_df[TARGET])

model_pipeline.fit(X_train, y_train, classifier__sample_weight=sample_weights)
#train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train the model
# This automatically runs the data through the scaler/encoder, then trains the FFN
print("Training model...")
model_pipeline.fit(X_train, y_train)
print("Training complete.\n")



Training model...
Training complete.



In [39]:
#pred on test set
y_pred = model_pipeline.predict(X_test)

y_test = y_test.astype(int)
y_pred = y_pred.astype(int)

target_names = [str(cls) for cls in label_encoder.classes_]

print("Classification Report:")
print(classification_report(
    y_test, 
    y_pred, 
    target_names=target_names
))

print(f"Overall Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Classification Report:
              precision    recall  f1-score   support

         0.0       0.80      1.00      0.89   1013082
         1.0       0.21      0.00      0.00     92894
         2.0       0.31      0.00      0.00     49885
         3.0       0.00      0.00      0.00     31087
         4.0       0.00      0.00      0.00     21001
         5.0       0.52      0.19      0.28     73368

    accuracy                           0.80   1281317
   macro avg       0.31      0.20      0.20   1281317
weighted avg       0.69      0.80      0.72   1281317

Overall Accuracy: 0.7980
